# Best encoding settings

The aim here is to train a model on existing data in order to estimate the FFmpeg parameters that optimise any arbitrary criterion.

In [1]:
import pathlib
import subprocess
import tempfile

import cutcutcodec

from mendevi.encode import get_transcode_cmd
from mendevi.models import Solver
from mendevi.models.nn import EnergyNN, QualityNN

In [2]:
dataset = "scatter.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Train the model

Before making predictions, you first need to train a model.

### Quality model

Quality does not depend on the machine. We therefore produce just one large, unique model.

In [3]:
# declaration
quality_model = QualityNN(
    output_labels=["log_size_per_frame", "psnr", "ssim"],
    weights="quality",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=15000,  # number of training iterations, default 10000
    learning_rate=1e-5,  # adam gradient decay learning rate, default 1e-4
    total=50_000,  # number of model parameters, default based on number of observations
)

In [4]:
# train (on cpu encoders only)
if not quality_model.is_fit():
    quality_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

Fit 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 17853 observations
█ █ Save model into /home/robin/.cache/mendevi/quality_29_3_50000.pk.xz...                                            
█ █ █ 29.80 ms
█ █ 13 min 8 s
█ 13 min 38 s
Save model into /home/robin/.cache/mendevi/quality.pk.xz...
█ 33.69 ms


In [5]:
# validation
quality_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Quality estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 4551 observations
█ █ 59.86 ms
█ 12.27 s
MAPE of log_size_per_frame: 1.80%
RMS  of log_size_per_frame: 0.09483
MAPE of psnr: 2.37%
RMS  of psnr: 1.644
MAPE of ssim: 1.34%
RMS  of ssim: 0.02013
MAPE of size_per_frame: 16.50%
RMS  of size_per_frame: 4.967e+04


### Energy model

Energy depends ont the machine. We therefore produce several models, one per hostname.

In [6]:
energy_model = EnergyNN(
    weights="energy",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=8000,  # number of training iterations, default 10000
    learning_rate=1e-4,  # adam gradient decay learning rate, default 1e-4
    total=2000,  # number of model parameters, default based on number of observations
)

Load model from /home/robin/.cache/mendevi/energy.pk.xz...
█ 23.04 ms


In [7]:
if not energy_model.is_fit():
    energy_model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val} and encoder_family == 'cpu'",
    )

In [8]:
# validation
energy_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val} and encoder_family == 'cpu'")

Validate 'Encoding energy estimation based on a neuronal network'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster hostname=abacus26-1.rennes.grid5000.fr...
█ █ this cluster contains 1687 observations
█ █ 23.59 ms
█ Validate on cluster hostname=dahu-12.grenoble.grid5000.fr...
█ █ this cluster contains 676 observations
█ █ 10.24 ms
█ Validate on cluster hostname=gemini-2.lyon.grid5000.fr...
█ █ this cluster contains 256 observations
█ █ 5.72 ms
█ Validate on cluster hostname=nova-12.lyon.grid5000.fr...
█ █ this cluster contains 476 observations
█ █ 7.96 ms
█ Validate on cluster hostname=paradoxe-32.rennes.grid5000.fr...
█ █ this cluster contains 618 observations
█ █ 10.73 ms
█ Validate on cluster hostname=suzanne...
█ █ this cluster contains 405 observations
█ █ 7.71 ms
█ Validate on cluster hostname=yeti-2.grenoble.grid5000.fr...
█ █ this cluster contains 433 observations
█ █ 7.95 ms
█ 31.20 s
MAPE of log_act_duration_per_frame: 22.02%
RMS  of log_act_duration_per_frame

## Use the model to perform predictions

The idea is to run a huge number of predictions using the trained model in order to explore a very large range of scenarios.

In [9]:
# example best compression for 40 dB

def best_dbrate(**kwd: dict[str]) -> float:
    """Loss function of debit rate distortion around 35 dB."""
    db_loss = 0.0 if 39 < kwd["psnr"] < 41 else ((kwd["psnr"]-40)**2)-1
    rate_loss = kwd["log_size_per_frame"]
    return db_loss + 1e-2*rate_loss

solver = Solver(quality_model, best_dbrate)
tested_combinations, loss, cmd = solver.solve(
    video,
    encoder=["libx264", "libx265", "libaom-av1", "libsvtav1", "libvvenc"],
    return_cmd=True,
)

Solve...
█ Predict 'Quality estimation based on a neuronal network'...
█ █ check args
█ █ vectorize args
█ █ predict
█ █ Predict for cluster ...
█ █ █ this cluster contains 990 observations
█ █ █ 11.30 ms
█ █ 18.18 ms
█ Result for loss = 0.03285:
█ █ effort            : slow
█ █ encoder           : libx265
█ █ height            : 720
█ █ log_size_per_frame: 3.285
█ █ mode              : crf
█ █ nb_frames         : 294
█ █ psnr              : 39.62
█ █ quality           : 0.6176
█ █ rms_sobel         : 0.1544
█ █ rms_time_diff     : 0.08223
█ █ spatial_dct       : 0.05345
█ █ ssim              : 0.9529
█ █ temporal_dct      : 0.01171
█ █ width             : 1280
█ ffmpeg -hide_banner -y -loglevel verbose -c:v libvpx-vp9 -i /home/robin/documents/programation/python/en_cours/cutcutcodec_git/cutcutcodec/media/video/intro.webm -c:v libx265 -crf 31.5 -preset slow -tune ssim -x265-params frame-threads=8:pools=8:wpp=1 -map 0:v:0 /tmp/intro_optimal.mp4
█ mendevi probe /tmp/intro_optimal.mp4 -r 

In [10]:
# perform the conversion
out_video = pathlib.Path(tempfile.gettempdir()) / "video_test.mp4"
cmd.output = [*cmd.output[:-1], str(out_video)]
output, activity = cmd.run()

In [11]:
# final verification
out = subprocess.run(
    ["mendevi", "probe", str(out_video), "-r", str(video), "-d", str(out_video.with_suffix(".db"))],
    check=True,
)

Parse configuration...
█ video     : /tmp/video_test.mp4
█ database  : /tmp/video_test.db (just created)
█ ref       : intro.webm
█ lpips_alex: no
█ lpips_vgg : no
█ psnr      : yes
█ rms_sobel : yes
█ rms_time_diff: yes
█ spatial_dct: yes
█ ssim      : yes
█ temporal_dct: yes
█ uvq       : no
█ vif       : no
█ vmaf      : no
Probe 1/2...                                                     
█ Get properties of intro.webm...
█ █ eotf         : bt1361e, bt1361
█ █ duration     : 9.81 s
█ █ name         : intro.webm
█ █ frames       : 294 frames
█ █ codec        : vp9
█ █ fps          : 29.97 Hz
█ █ size         : 0.50 MB
█ █ pix fmt      : yuv420p
█ █ width        : 1280 pixels
█ █ gamut        : bt709
█ █ range        : tv
█ █ height       : 720 pixels
█ █ rms_sobel    : 0.15 +/- 0.07                                                                                                                                                 
█ █ rms_time_diff: 0.08 +/- 0.05
█ █ spatial_dct  : 0.05 +/